# Treewidth of protein contact graphs

Exact optimization of a Potts model is exponential in the treewidth of its interaction graph, not in the number of residues. This notebook measures what controls that treewidth, first on synthetic graphs, then on real proteins.

Treewidth is NP-hard to compute exactly, so I use NetworkX's min-fill-in heuristic (an upper bound). Reusable code is in [`src/`](../src); the synthetic part needs no external data, the protein part needs PDB files from RCSB.

## Part 1: synthetic graphs of the same size, different structure

In [ ]:
import sys; sys.path.append('..')
from src.graphs import reference_graphs, estimate_treewidth

for name, g in reference_graphs(20).items():
    print(f'{name:6} edges={g.number_of_edges():3d}  treewidth={estimate_treewidth(g)}')

Path 1, cycle 2, grid 4, clique 19 — same 20 nodes, treewidth set by structure.

## Adding random edges to a path
Hold the node count fixed and add random chords; average over 20 trials.

In [ ]:
import random, numpy as np, pandas as pd
from src.graphs import path_with_random_edges

rng = random.Random(42)
rows = []
for ne in range(0, 21, 2):
    ws = [estimate_treewidth(path_with_random_edges(20, ne, rng)) for _ in range(20)]
    rows.append({'extra_edges': ne, 'mean_tw': np.mean(ws), 'sd_tw': np.std(ws)})
pd.DataFrame(rows)

Treewidth grows about linearly with edges added. Connectivity, not size, is what moves it.

## Part 2: real protein contact graphs
An edge joins two residues whose Cα atoms are within a cutoff. Needs the PDB files locally (download from RCSB first).

In [ ]:
from src.contacts import protein_stats
import pandas as pd

# point PDB_DIR at a folder with 1CRN.pdb, 1UBQ.pdb, 1AKE.pdb, 1TIM.pdb, 1GC1.pdb
PDB_DIR = '../pdbs'
jobs = [('1CRN','A'), ('1UBQ','A'), ('1AKE','A'), ('1TIM','A'), ('1GC1','G')]
pd.DataFrame([protein_stats(f'{PDB_DIR}/{p}.pdb', p, chain_ids=[c], cutoff=15.0)
              for p, c in jobs])

Density doesn't track treewidth (1GC1 is the least dense but near the top). Average degree does: across a cutoff sweep on 1GC1, treewidth and average degree correlate at r = 0.96.

## The main result: long-range contacts drive treewidth
Keep only contacts up to a sequence-separation cap |i − j| and drop the rest.

In [ ]:
from src.contacts import contact_graph

g = contact_graph(f'{PDB_DIR}/1GC1.pdb', chain_ids=['G'], cutoff=15.0)
print('full graph treewidth:', estimate_treewidth(g))
capped = g.copy()
capped.remove_edges_from([(i, j) for i, j in g.edges() if abs(i - j) > 12])
print('|i-j| <= 12 treewidth:', estimate_treewidth(capped))

## Summary
- At fixed size, treewidth is set by structure: connectivity, mainly average degree.
- For proteins, density doesn't predict treewidth but average degree does (r = 0.96).
- Long-range contacts are the driver: on 1GC1, capping sequence separation at 12 drops treewidth from 114 to 12.
- This is why restricting couplings to a sequence-neighbor window makes exact Potts optimization tractable (see the beyond-native-recovery repo). This notebook measures what that restriction costs.